# 07.3 — KV-transfer, layerwise effect (OVReader)

Like 07.1/07.2 but the intervention is **KV-transfer** (transplant the SRC task-list keys+values into the
DST goal heads over the aligned task span), not q-transfer. We read the **layerwise** effect on the
**patched DST run** with OVReader (residual OV-write → GoalScope read-out per probe layer), with options
to also read the SRC run and the clean DST run. KV gaze viz mirrors `6_disent_from_attn_out` (patched
q_dst × k_src on the SRC-task span, vs clean DST).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "3,7"   # GPUs with room (avoid the eval jobs)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(".."); sys.path.append("../..")

import torch, baukit
from contextlib import nullcontext, ExitStack
from tqdm.auto import tqdm
from rich import print as rich_print
from rich.markup import escape
from rich.table import Table
from rich.panel import Panel
from rich.text import Text

import src.diff.functional as P
from src.diff.functional import *          # cache_src_proj, headwise_kv_patch, heads_per_layer, greedy,
                                            # align_kv_spans, token_span, make_patch_interface, proj_patch_hooks,
                                            # boost_head_ov, pivot_attn, show_delta_tokens, goalscope_readout, _proj_name …
import src.diff.data as D
import scripts.evaluation.eval_ov_reader as R
from scripts.evaluation.eval_goalscope import hard_eval, CAUSAL_PREFIX
from src.attention import get_attention_matrices
from src.goals.goalscope import OVReader, head_additive_generate, _build_goalscope_prompt, _build_causality_prompt
from src.tokens import prepare_input

os.chdir(os.path.abspath(os.path.join(os.path.dirname(R.__file__), "..", "..")))
torch.set_grad_enabled(False)

MODEL = "Qwen/Qwen2.5-Coder-32B-Instruct"
P.setup(MODEL)                  # loads model (eager) + head masks; binds P globals (mt, GROUP, GOAL_HEADS, …)
mt = P.mt

2026-09-10 19:40:27 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [3]:
# ── config (07.1-style knobs + KV-transfer knobs) ──
DATATYPE       = "list[str]"    # int | str | list[int] | list[str]
SRC_LANG       = "python"       # per-side language (cross-lang kv: align_kv_spans pad-aligns)
DST_LANG       = "python"
MODE           = "faithfulness" # faithfulness | causality
LAYERS         = list(range(36, 62, 2))   # OVReader probe layers (residual after each)
STEER_BAND     = (35, 46)       # GoalScope steer band
ALPHA_OV       = 0.15           # OVReader read-out (additive_norm)
OV_BOOST       = 2.0            # OV boost on PATCH_HEADS during the KV-transfer generation/readout
KV_STUB        = "var_eq"       # pivot stub: "var_eq" (c =) | "indent"
FREEZE_DEST_Q  = False          # also pin PATCH_HEADS query at the pivot to clean-DST (kv-only influence)
MAXNEW         = 64             # OVReader read-out length
GEN_NEW        = 96             # kv-transfer generation length
SEED           = 0
SHOW_SRC       = False           # also OVReader the SRC run
SHOW_CLEAN_DST = False           # also OVReader the clean DST run
GOAL_HEADS    = P.GOAL_HEADS    # heads used by GoalScope / gaze

PATCH_HEADS   = P.GOAL_HEADS    # heads whose k/v are transplanted (and OV-boosted)
READOUT_HEADS = P.GOAL_HEADS    # heads used by OVReader / GoalScope / gaze


# OPTIONAL — use the top-250 goal-head set (patch + readout), which gave near-perfect kv transfer.
import json
_p = "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json"
GOAL_HEADS_250 = [tuple(h) for h in json.load(open(_p))]

# OVERRIDE-------------------------------------------------------------------------------
# GOAL_HEADS = GOAL_HEADS_250
LAYER_BOTTLENECK = [42]
# READOUT_HEADS = [(l, h) for (l, h) in GOAL_HEADS if l >= min(LAYER_BOTTLENECK)]
# PATCH_HEADS = [(l, h) for (l, h) in GOAL_HEADS if l in LAYER_BOTTLENECK] 
PATCH_HEADS = [(l, h) for (l, h) in GOAL_HEADS if l > min(LAYER_BOTTLENECK)] 
# PATCH_HEADS = [(l, h) for h in range(mt.n_heads) for l in range(42, 54, 1)] 
# PATCH_HEADS = GOAL_HEADS
print(f"{len(GOAL_HEADS)} goal heads  |  {len(PATCH_HEADS)} patch heads  |  {len(READOUT_HEADS)} readout heads")


ov  = OVReader(mt, READOUT_HEADS)
GSP = _build_causality_prompt(mt) if MODE == "causality" else None
P.gsp = GSP
PROBE  = prepare_input(prompts=(GSP or _build_goalscope_prompt(mt)), tokenizer=mt)
PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
rich_print(f"heads={len(PATCH_HEADS)}  layers={LAYERS}  mode={MODE}  {DATATYPE}  {SRC_LANG}->{DST_LANG}")

122 goal heads  |  85 patch heads  |  122 readout heads


Building OV readers:   0%|          | 0/122 [00:00<?, ?it/s]

heads=85  layers=[36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60]  mode=faithfulness  list  python->python

In [4]:
# ── nb6-local helpers (verbatim): instance builder, list body, var_eq stub, prompt tweak, OV boost, span viz ──
BANK = D.QTASKS[DATATYPE]; BY = {p.id: p for p in BANK.predicates}
# T1, T2, T3, T4, T5 = list(BANK.predicates)[0].id, *[p.id for p in list(BANK.predicates)[1:5]]
T1, T2, T3, T4, T5 = "total_length", "total_uppercase_letters", "has_palindrome", "count_starts_vowel", "all_unique_lengths"


def mk(pred_ids, progress, target, lang="python"):
    task = D.dataclasses.replace(BANK, id="xtask", predicates=tuple(BY[i] for i in pred_ids))
    return D.QTransferInstance(task=task, var_name=BANK.var_names[0], presentation="listed",
        src_lang=lang, dst_lang=lang, src_progress=progress, dst_progress=progress,
        expected_next=target, naming="letters", meta={"datatype": DATATYPE})

def list_body(inst):
    return "\n".join(f"({i+1}) {p.description}" for i, p in enumerate(inst.task.predicates))

def get_prompt(question, prefill, ctx=None):     # notebook-local: nudge sequential order
    return P.get_prompt(question.replace("computes:", "computes in sequential order:"), prefill, ctx=ctx)

def stub_parts(inst, side, stub):
    q, pf = (inst.src_parts() if side == "src" else inst.dst_parts())
    if stub != "var_eq":
        return q, pf
    lang = inst.src_lang if side == "src" else inst.dst_lang
    prog = inst.src_progress if side == "src" else inst.dst_progress
    var = inst.var_of(inst.expected_next)
    body = pf.rstrip("\n ")
    if lang == "cpp":
        for pid in prog:
            body = D._cpp_autoify(body, inst.var_of(pid))
        return q, body + f"\n    auto {var} ="
    return q, body + f"\n    {var} ="

def ov_boost(positions=(-1,)):
    return boost_head_ov(mt, PATCH_HEADS, OV_BOOST, positions=positions) if OV_BOOST != 1.0 else nullcontext()

def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids); pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set(); kv = set(range(*kv_span)) if kv_span else set()
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:   s = PO + s + PC
        elif i in kv:     s = KO + s + KC
        elif i in task:   s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

In [5]:
# ── build the KV-transfer contrastive pair: shared progress, different task → transplant SRC task-list K,V ──
k_src = mk([T1, T2, T3], [T1, T2], T3)     # SRC task/target = T3
k_dst = mk([T2, T4, T1], [T2, T1], T4)     # DST native frontier = T4
k_src, k_dst = k_dst, k_src # swap
k_src.src_lang = SRC_LANG; k_dst.dst_lang = DST_LANG

_q, _pf = stub_parts(k_src, "src", KV_STUB); ksrc_prompt = get_prompt(_q, D._finalize_prefill(_pf))
_q, _pf = stub_parts(k_dst, "dst", KV_STUB); kdst_prompt = get_prompt(_q, D._finalize_prefill(_pf))
src_text, dst_text = list_body(k_src), list_body(k_dst)         # transplant span = task-list body
src_span0 = token_span(ksrc_prompt, src_text); dst_span0 = token_span(kdst_prompt, dst_text)
ksrc_tok, kdst_tok, src_pos, dst_pos, src_span, dst_span = align_kv_spans(
    mt, ksrc_prompt, kdst_prompt, src_span0, dst_span0, mode="pad", src_text=src_text, dst_text=dst_text)
kpiv_src = ksrc_tok["input_ids"].shape[1] - 1
kpiv_dst = kdst_tok["input_ids"].shape[1] - 1
SRC_TARGET = k_src.expected_next; DST_NATIVE = k_dst.expected_next
rich_print(f"SRC target {SRC_TARGET} = [green]{escape(BY[SRC_TARGET].description)}[/]   "
           f"DST native {DST_NATIVE} = [magenta]{escape(BY[DST_NATIVE].description)}[/]")
show_spans(ksrc_tok, f"SRC · cached-from KV window {tuple(src_span)} · pivot@{kpiv_src}",
           task_span=src_span, kv_span=(src_pos[0], src_pos[-1] + 1), piv=-1)
show_spans(kdst_tok, f"DST · patched-to KV window {tuple(dst_span)} · pivot@{kpiv_dst}",
           task_span=dst_span, kv_span=(dst_pos[0], dst_pos[-1] + 1), piv=-1)

SRC target count_starts_vowel = the number of words that start with a vowel   DST native has_palindrome = whether 
any word is a palindrome

╭─ SRC · cached-from KV window (66, 103) · pivot@152 ─────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ (1) the total number of uppercase letters across all words\n                                                    │
│ (2) the number of words that start with a vowel\n                                                               │
│ (3) the total number of characters across all words\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c = sum(len(w) for w in words)\n                                                                            │
│     b =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · patched-to KV window (66, 103) · pivot@152 ──────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes in    │
│ sequential order:\n                                                                                             │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the total number of characters across all words\n                                                           │
│ (2) the total number of uppercase letters across all words\n                                                    │
│ (3) whether any word is a palindrome\n                                                                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(len(w) for w in words)\n                                                                            │
│     b = sum(1 for w in words for c in w if c.isupper())\n                                                       │
│     c =                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
# ── KV-transfer patch setup + residual/readout helpers (generation & read-outs live in the run cells) ──
kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(P._layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(ksrc_tok, src_pos, sorted(kvheads), "k")
v_cache = cache_src_proj(ksrc_tok, src_pos, sorted(kvheads), "v")
qh = heads_per_layer(PATCH_HEADS, "q"); qQ = sorted({l for l, _ in PATCH_HEADS})
if FREEZE_DEST_Q:
    q_dst_clean = cache_src_proj(kdst_tok, [kpiv_dst], qQ, "q")
    qpf = [dict(which="q", cache=q_dst_clean, heads=qh, positions=[kpiv_dst])]
    qdc = [dict(which="q", cache=q_dst_clean, heads=qh, positions=[-1])]
else:
    qpf = qdc = None

def kv_patch_stack():
    """ExitStack of the exact KV-transfer intervention (headwise k/v routing + OV boost + optional Q-freeze)."""
    es = ExitStack()
    es.enter_context(headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, dst_pos))
    es.enter_context(ov_boost())
    if FREEZE_DEST_Q:
        es.enter_context(proj_patch_hooks(mt, qpf))
    return es

def kv_patched_iface():
    def _if(mt, inputs, patches=None, model_kwargs=None):
        with kv_patch_stack():
            return mt._model(**dict(inputs), **dict(model_kwargs or {}))
    return _if

def extract_resid(tok, pivot, layers, patch_stack=None):
    """Residual after each layer at `pivot`, optionally under a patch ExitStack (e.g. the KV-transfer)."""
    ln = lambda l: mt.layer_name_format.format(l)
    unt = lambda x: x[0] if isinstance(x, tuple) else x
    ctx = patch_stack if patch_stack is not None else nullcontext()
    with torch.no_grad(), ctx, baukit.TraceDict(mt._model, [ln(l) for l in layers], retain_output=True) as tr:
        mt._model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
    return {l: unt(tr[ln(l)].output)[0, pivot].detach().float().cpu() for l in layers}

def ov_layerwise(resid_by_layer):
    """Print the full OVReader read-out per probe layer — BATCHED (one head_additive_generate over all layers)."""
    outs = ov.apply_probe(resid_by_layer, mode=MODE, steer_layers=list(range(*STEER_BAND)),
                          max_new_tokens=MAXNEW, alpha=ALPHA_OV)
    for L in LAYERS:
        print(f"    OVReader L={L:2d}: {escape(PREFIX + outs[L])}")

print("KV patch ready · PATCH_HEADS == READOUT_HEADS:", set(PATCH_HEADS) == set(READOUT_HEADS))

KV patch ready · PATCH_HEADS == READOUT_HEADS: False


In [7]:
# ── SOURCE run: SRC generation + read-outs (goalscope @ pivot, OVReader layerwise) ──
rich_print(f"[bold]=== SOURCE run · target {SRC_TARGET}={BY[SRC_TARGET].description!r} ===[/]")
rich_print(f"[dim]SRC generation:[/]\n{escape(greedy(ksrc_tok, max_new=GEN_NEW))}")
goalscope_readout("  goalscope . SRC @ pivot", ksrc_tok, kpiv_src, heads=READOUT_HEADS)
if SHOW_SRC:
    ov_layerwise(extract_resid(ksrc_tok, kpiv_src, LAYERS))

=== SOURCE run · target count_starts_vowel='the number of words that start with a vowel' ===

SRC generation:
 sum(1 for w in words if w[0].lower() in 'aeiou')
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


  goalscope . SRC @ pivot  pos=152 ' =' => I'm checking if the character is a vowel.

In [8]:
# ── CLEAN DST run: generation + read-outs + gaze (pivot -> DST-task span) ──
rich_print(f"[bold]=== CLEAN DST run · native {DST_NATIVE}={BY[DST_NATIVE].description!r} ===[/]")
rich_print(f"[dim]clean DST generation:[/]\n{escape(greedy(kdst_tok, max_new=GEN_NEW))}")
goalscope_readout("  goalscope . clean DST @ pivot", kdst_tok, kpiv_dst, heads=READOUT_HEADS)

if SHOW_CLEAN_DST:
    ov_layerwise(extract_resid(kdst_tok, kpiv_dst, LAYERS))
# ATTN VIS: clean pivot -> DST-task span (value-weighted, READOUT_HEADS)
dst_task_toks = [mt.tokenizer.decode([int(kdst_tok["input_ids"][0][j])]) for j in dst_pos]
a_clean = pivot_attn(get_attention_matrices(kdst_tok, mt, value_weighted=True),
                     list(READOUT_HEADS), dst_pos, kpiv_dst)

show_delta_tokens(dst_task_toks, a_clean, "clean gaze . pivot->DST-task span (value-weighted, READOUT_HEADS)")

=== CLEAN DST run · native has_palindrome='whether any word is a palindrome' ===

clean DST generation:
 any(w == w[::-1] for w in words)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope . clean DST @ pivot  pos=152 ' =' => I'm checking if any element in the list is a palindrome.

clean gaze . pivot->DST-task span (value-weighted, READOUT_HEADS)   (green +Δ / red −Δ, |Δ|max=1.341)

In [9]:
# ── PATCHED (kv-transfer) DST run: generation + read-outs + gaze (q_dst x k_src on SRC-task span) ──
rich_print(f"[bold]=== PATCHED DST run (kv-transfer) · expect SRC {SRC_TARGET}={BY[SRC_TARGET].description!r} ===[/]")
with kv_patch_stack():
    kvgen = greedy(kdst_tok, prefill_specs=qpf, decode_specs=qdc, n_steer=32, max_new=GEN_NEW)
rich_print(f"[salmon1]kv-transfer generation:[/]\n{escape(kvgen)}")
goalscope_readout("  goalscope . patched DST @ pivot", kdst_tok, kpiv_dst, heads=READOUT_HEADS, iface=kv_patched_iface())
ov_layerwise(extract_resid(kdst_tok, kpiv_dst, LAYERS, kv_patch_stack()))

# ATTN VIS: patched q_dst x k_src (value-weighted) laid on the SRC-task span. Assumes PATCH == READOUT heads.
if set(PATCH_HEADS) != set(READOUT_HEADS):
    rich_print("[bright_red]WARNING: PATCH_HEADS != READOUT_HEADS -- the patched gaze reads heads whose k/v were "
               "not transplanted, so q_dst x k_src on the SRC span is not meaningful.[/bright_red]")
kvh_map = heads_per_layer(PATCH_HEADS, "k", layers=set(P._layers_of(PATCH_HEADS)))
kv_viz = [dict(which="k", cache=k_cache, heads=kvh_map, positions=dst_pos),
          dict(which="v", cache=v_cache, heads=kvh_map, positions=dst_pos)]
if FREEZE_DEST_Q:
    kv_viz = kv_viz + qpf
src_task_toks = [mt.tokenizer.decode([int(ksrc_tok["input_ids"][0][j])]) for j in src_pos]
a_patched = pivot_attn(get_attention_matrices(kdst_tok, mt, value_weighted=True,
                       patch_interface=make_patch_interface(mt, kv_viz)),
                       list(READOUT_HEADS), dst_pos, kpiv_dst)
show_delta_tokens(src_task_toks, a_patched,
                  f"patched gaze . pivot->SRC-task (q_dst{'[frozen]' if FREEZE_DEST_Q else ''} x k_src, value-weighted, READOUT_HEADS)")

=== PATCHED DST run (kv-transfer) · expect SRC count_starts_vowel='the number of words that start with a vowel' ===

kv-transfer generation:
 any(w.lower().startswith(('a', 'e', 'i', 'o', 'u')) and w.lower() == w.lower()[::-1] for w in words)
    return a, b, c
</code>

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope . patched DST @ pivot  pos=152 ' =' => I'm checking if any element in the list satisfies the condition 
of being equal to its reverse (i.e., a palindrome).

    OVReader L=36: I'm checking if there's a condition that evaluates to `True` for all elements in an iterable, specifically if there's a condition that returns `True` for every element in the list. This is often done with a generator expression inside the `all()` function, which checks for a condition that must be met by
    OVReader L=38: Sure, I can help with that. Please provide the specific line or snippet of code where you want to check for a condition, and I'll assist you in identifying if there's a condition checking for a specific element or logic to find if any element satisfies a given condition.

For example, if you have a list of
    OVReader L=40: Sure, I can check if there's a line in your code that evaluates to `True` for any element in a list using a lambda function with `any()`. Here's a concise way to do it:

```python
any(lambda x: x == x[::-1] and x != '' for x in ['
    OVReader L=42: I'm checking if any element in the list satisfies the condition that it is equa

WARNING: PATCH_HEADS != READOUT_HEADS -- the patched gaze reads heads whose k/v were not transplanted, so q_dst x 
k_src on the SRC span is not meaningful.

patched gaze . pivot->SRC-task (q_dst x k_src, value-weighted, READOUT_HEADS)   (green +Δ / red −Δ, |Δ|max=0.628)